In [ ]:
from google.colab import userdata
GIT_TOKEN=userdata.get('GITHUB_TOKEN')

In [ ]:
# Install the Vela compiler toolchain
!pip install ethos-u-vela

# Secure clone and pip install your custom codebase using GitHub Token credentials
import os
GIT_USERNAME = "bencejdanko"
GIT_REPO = f"github.com/{GIT_USERNAME}/fomo-people-counting.git"

!git clone https://{GIT_USERNAME}:{GIT_TOKEN}@{GIT_REPO}
%cd /content/fomo-people-counting
!pip install -e .

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from fomo_core.annotation import COCOPersonAnnotator
from fomo_core.pipeline import FomoDatasetBuilder
from fomo_core.model import build_and_compile_fomo
from fomo_core.quantization import convert_to_int8_tflite

# 1. Pipeline Setup
repo_id = "bdanko/overhead-people-rgb"
# Merge preserved source labels such as Man, Woman, Person, ero, and 0 into one person class.
annotator = COCOPersonAnnotator(category_id=0, category_name="person", merge_all_categories=True)
builder = FomoDatasetBuilder(annotator=annotator)

# Load the entire dataset locally, then make a deterministic 90/10 train/validation split.
raw_dataset = load_dataset(repo_id, split="train")
split_dataset = raw_dataset.train_test_split(test_size=0.10, seed=42, shuffle=True)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]
print(f"Rows: train={len(train_dataset)}, val={len(val_dataset)}")

# 2. Linear Dataset Ingestion
def build_arrays(hf_dataset, split_name):
    print(f"Extracting {split_name} features...")
    images, targets = [], []
    for example in hf_dataset:
        img_arr, grid_arr = builder.process_example(example)
        images.append(img_arr)
        targets.append(grid_arr)
    return np.array(images), np.array(targets)

X_train, y_train = build_arrays(train_dataset, "train")
X_val, y_val = build_arrays(val_dataset, "validation")
print("Training tensors:", X_train.shape, y_train.shape)
print("Validation tensors:", X_val.shape, y_val.shape)

# 3. Sanity Visualization
def show_target_overlay(images, targets, count=4):
    count = min(count, len(images))
    fig, axes = plt.subplots(1, count, figsize=(4 * count, 4))
    if count == 1:
        axes = [axes]

    for ax, image, target in zip(axes, images[:count], targets[:count]):
        display_img = ((image + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
        ax.imshow(display_img)
        object_cells = np.argwhere(np.argmax(target, axis=-1) == 1)
        for grid_y, grid_x in object_cells:
            x = (grid_x + 0.5) * builder.input_size / builder.grid_size
            y = (grid_y + 0.5) * builder.input_size / builder.grid_size
            ax.scatter(x, y, c="red", s=35, marker="x")
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()

show_target_overlay(X_train, y_train)

# 4. TF Data Handlers
batch_size = 32
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (
    train_ds
    .shuffle(min(len(X_train), 2000), reshuffle_each_iteration=True)
    .map(builder.synchronized_augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = (
    val_ds
    .batch(batch_size)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# 5. Engine Run
model = build_and_compile_fomo()
model.fit(train_ds, validation_data=val_ds, epochs=15)

# 6. INT8 Calibration
convert_to_int8_tflite(model, X_train)

# 7. Call the Vela compiler toolchain via bash
!vela fomo_production_int8.tflite \
  --accelerator-config ethos-u55-256 \
  --config configs/default_vela.ini \
  --memory-mode Shared_Sram \
  --system-config Ethos_U55_High_End_Embedded \
  --output-dir vela_output \
  --optimise Size
